# Annex 96 Common Exercise 1 – Quick Start Notebook

This notebook provides a minimal end-to-end setup for:


1. Cloning the CE1 GitHub repository
2. Understanding the TX & VT datasets
3. Visualizing the daily reference district load
4. Running a simple baseline battery RBC controller
5. Evaluating KPIs

If you run into issues, please reach out on the Annex 96 Slack (Common Exercise channel).


# 1. Install Required Dependencies

This Common Exercise uses a **specific version of CityLearn included inside this repository**.

👉 **Do NOT install CityLearn using `pip install citylearn`.**  
This will install a different version that is **not compatible** with the Common Exercise and will cause errors.

Instead, please install all dependencies directly from the repository’s `requirements.txt` file.





### Clone the Common Exercise repository


In [1]:
# !git clone https://github.com/kkaspar10/annex96_common_exercise_1.git

### Install required Python packages

In [2]:
# !pip install -U ipykernel

In [ ]:
# !pip install -r requirements.txt

/bin/bash: line 1: pip: command not found


## 2. Dataset Overview (TX vs VT)

The Common Exercise uses two scenarios:

### 🔵 Texas (TX) – cooling dominated  
- Training: **June**  
- Testing: **July**  
- Higher solar generation, strong cooling loads.

### 🔵 Vermont (VT) – heating dominated  
- Training: **January**  
- Testing: **February**  
- High heating loads, low PV in winter.

Each climate folder contains:

- `schema.json`  
- 25 ResStock buildings  
- Pre-computed district reference load  
- All data handled automatically by CityLearn




In [4]:
from pathlib import Path
import os

# Where the notebook actually lives
NOTEBOOK_DIR = Path(os.getcwd())
print("Notebook directory:", NOTEBOOK_DIR)

# Go one level up to the repo root
BASE_DIR = NOTEBOOK_DIR.parent
print("Repository root:", BASE_DIR)

# Choose climate: TX or VT
CLIMATE = "VT"  # or "VT"

DATASET_DIR = BASE_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

print("\nClimate:", CLIMATE)
print("Dataset directory:", DATASET_DIR, " | exists:", DATASET_DIR.exists())
print("Schema path:", SCHEMA_PATH, " | exists:", SCHEMA_PATH.exists())


Notebook directory: /home/yue/EthanZhu/annex96_common_exercise_1/notebooks
Repository root: /home/yue/EthanZhu/annex96_common_exercise_1

Climate: VT
Dataset directory: /home/yue/EthanZhu/annex96_common_exercise_1/data/datasets/annex96_ce1_vt_neighborhood  | exists: True
Schema path: /home/yue/EthanZhu/annex96_common_exercise_1/data/datasets/annex96_ce1_vt_neighborhood/schema.json  | exists: True


## 3. District Load Target

Each day, the Annex 96 CE defines a **constant district load target** equal to the average aggregated demand of all buildings for the each day.

Controllers attempt to track this reference profile.

Below is the reference load plot for the selected climate (TX or VT).

![District Target](../assets/images/district_load_target.png)



In [5]:
import pandas as pd

district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values

## 4. Run a Simple Battery RBC Controller

This baseline:

- Uses the built-in `BasicRBC` controller  
- Controls all buildings through a **central agent**  
- Produces reference KPIs to compare against learning-based methods  


In [6]:
# === Notebook cell: CityLearn local outside working directory ===
import sys
from pathlib import Path
import traceback

# --- 1) set main paths (modify according to your structure) ---
NOTEBOOK_ROOT = Path.cwd()  # working directory of the notebook
print("Working directory:", NOTEBOOK_ROOT)

# Parent folder containing the 'citylearn' package
CITYLEARN_PATH = Path("../")   # adjust if citylearn is in a different location
sys.path.insert(0, str(CITYLEARN_PATH))
print("Added citylearn path to sys.path:", CITYLEARN_PATH)

name = 'tx'
# Folder containing dataset / data
DATASET_DIR = Path(f"../data/datasets/annex96_ce1_{name}_neighborhood")  # modify according to your dataset

# Try to automatically find a schema.json under DATASET_DIR
SCHEMA_PATH = None
if DATASET_DIR.exists():
    for candidate in DATASET_DIR.rglob("schema.json"):
        SCHEMA_PATH = candidate
        break

# Fallback: search for schema.json in the notebook root
if SCHEMA_PATH is None:
    for candidate in NOTEBOOK_ROOT.rglob("schema.json"):
        SCHEMA_PATH = candidate
        break

if SCHEMA_PATH is None:
    raise FileNotFoundError("schema.json not found automatically. Please set SCHEMA_PATH manually.")

print("DATASET_DIR:", DATASET_DIR if DATASET_DIR.exists() else "(not found)")
print("SCHEMA_PATH:", SCHEMA_PATH)

# --- 2) import CityLearnEnv ---
try:
    from citylearn.citylearn import CityLearnEnv
    print("CityLearnEnv import OK")
except Exception as e:
    print("Error importing CityLearn. Traceback:")
    traceback.print_exc()
    raise ImportError("Cannot import 'citylearn' from the local folder.")

# --- 3) initialize the CityLearn environment ---
try:
    env = CityLearnEnv(schema=str(SCHEMA_PATH), root_directory=str(DATASET_DIR), central_agent=True)
except TypeError as e:
    print("TypeError in CityLearnEnv init:", e)
    print("Trying to create env passing only the schema (version compatibility)...")
    env = CityLearnEnv(schema=str(SCHEMA_PATH), central_agent=True)

# --- 4) reset the environment (Gym/Gymnasium compatible) ---
reset_ret = env.reset()
if isinstance(reset_ret, tuple):
    observations = reset_ret[0]
    info = reset_ret[1] if len(reset_ret) > 1 else {}
else:
    observations = reset_ret
    info = {}

# --- 5) diagnostic prints ---
print("\n--- CityLearn Diagnostics ---")
print("Number of buildings:", len(getattr(env, "buildings", [])))
if hasattr(env, "action_names") and env.action_names:
    print("Action names (first 10 of first building):", env.action_names[0][:10])
else:
    print("action_names not available or empty.")

if hasattr(env, "observation_names") and env.observation_names:
    print("Observation names (first 15 of first building):", env.observation_names[0][:15])
else:
    print("observation_names not available or empty.")

print("Type of observations:", type(observations))
try:
    if isinstance(observations, (list, tuple)) and len(observations) > 0:
        print("Observation preview (first 2 elements of first building):", observations[0][:2])
except Exception:
    pass

print("ENV initialized successfully ✅")



Working directory: /home/yue/EthanZhu/annex96_common_exercise_1/notebooks
Added citylearn path to sys.path: ..
DATASET_DIR: ../data/datasets/annex96_ce1_tx_neighborhood
SCHEMA_PATH: ../data/datasets/annex96_ce1_tx_neighborhood/schema.json
CityLearnEnv import OK

--- CityLearn Diagnostics ---
Number of buildings: 25
Action names (first 10 of first building): ['electrical_storage', 'cooling_device', 'electrical_storage', 'cooling_device', 'electrical_storage', 'cooling_device', 'electrical_storage', 'cooling_device', 'electrical_storage', 'cooling_device']
Observation names (first 15 of first building): ['month', 'hour', 'outdoor_dry_bulb_temperature', 'direct_solar_irradiance', 'outdoor_dry_bulb_temperature_predicted_1', 'outdoor_dry_bulb_temperature_predicted_2', 'outdoor_dry_bulb_temperature_predicted_3', 'direct_solar_irradiance_predicted_1', 'direct_solar_irradiance_predicted_2', 'direct_solar_irradiance_predicted_3', 'indoor_dry_bulb_temperature', 'non_shiftable_load', 'dhw_demand'

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))


from citylearn.agents.rbc import PITemperatureController as RBC


